In [1]:

import os
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# DAY 17: VERTICAL TEMPERATURE PROFILE ANALYSIS
# --------------------------------------------------

# Find the simulation results folder
folders = glob.glob("combined_mantle_results*")

if not folders:
    raise FileNotFoundError(
        "Run the combined mantle convection model first."
    )

result_folder = max(folders, key=os.path.getmtime)

files = glob.glob(os.path.join(result_folder, "*.npz"))

if not files:
    raise FileNotFoundError(
        "No .npz simulation files were found."
    )

output_folder = "day17_temperature_profiles"
os.makedirs(output_folder, exist_ok=True)

profiles = []

# --------------------------------------------------
# 1. LOAD SIMULATION RESULTS
# --------------------------------------------------

for filename in files:

    with np.load(filename) as data:

        if not all(
            key in data for key in ["ra", "z", "T"]
        ):
            print("Skipping incomplete file:", filename)
            continue

        Ra = float(np.asarray(data["ra"]).item())
        z = data["z"].copy()
        T = data["T"].copy()

    # Check the dimensions
    if T.ndim != 2 or T.shape[0] != len(z):
        print("Skipping incompatible temperature array:", filename)
        continue

    if not (
        np.all(np.isfinite(T))
        and np.all(np.isfinite(z))
    ):
        print("Skipping non-finite data:", filename)
        continue

    # Average temperature horizontally at each depth
    T_mean = np.mean(T, axis=1)

    # Temperature variation across each horizontal row
    T_std = np.std(T, axis=1)

    # Conductive reference profile
    T_conduction = 1.0 - z

    # Difference from the conductive reference
    T_anomaly = T_mean - T_conduction

    # RMS departure from conduction
    rms_anomaly = np.sqrt(
        np.mean(T_anomaly**2)
    )

    profiles.append({
        "Ra": Ra,
        "z": z,
        "T_mean": T_mean,
        "T_std": T_std,
        "T_conduction": T_conduction,
        "T_anomaly": T_anomaly,
        "rms_anomaly": rms_anomaly
    })

    print("\nRayleigh number:", Ra)
    print("RMS temperature departure:", rms_anomaly)
    print("Mean temperature at bottom:", T_mean[0])
    print("Mean temperature at top:", T_mean[-1])

if not profiles:
    raise ValueError("No valid temperature profiles were found.")

# Sort profiles by Rayleigh number
profiles.sort(key=lambda item: item["Ra"])

# --------------------------------------------------
# 2. PLOT VERTICAL TEMPERATURE PROFILES
# --------------------------------------------------

plt.figure(figsize=(7, 6))

# Plot the conductive reference
z_reference = profiles[0]["z"]

plt.plot(
    1.0 - z_reference,
    z_reference,
    "k--",
    linewidth=2,
    label="Pure conduction"
)

# Plot each simulated temperature profile
for item in profiles:

    plt.plot(
        item["T_mean"],
        item["z"],
        marker="o",
        markersize=3,
        label=f"Ra = {item['Ra']:g}"
    )

plt.xlabel("Normalized temperature")
plt.ylabel("Normalized height, z")
plt.title("Day 17: Vertical temperature profiles")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        output_folder,
        "day17_vertical_temperature_profiles.png"
    ),
    dpi=200
)

plt.close()

# --------------------------------------------------
# 3. PLOT TEMPERATURE ANOMALIES
# --------------------------------------------------

plt.figure(figsize=(7, 6))

for item in profiles:

    plt.plot(
        item["T_anomaly"],
        item["z"],
        marker="o",
        markersize=3,
        label=f"Ra = {item['Ra']:g}"
    )

plt.axvline(0, color="black", linestyle="--")

plt.xlabel("Temperature anomaly from conduction")
plt.ylabel("Normalized height, z")
plt.title("Day 17: Departure from conductive temperature")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        output_folder,
        "day17_temperature_anomalies.png"
    ),
    dpi=200
)

plt.close()

# --------------------------------------------------
# 4. PLOT HORIZONTAL TEMPERATURE VARIATION
# --------------------------------------------------

plt.figure(figsize=(7, 6))

for item in profiles:

    plt.plot(
        item["T_std"],
        item["z"],
        marker="o",
        markersize=3,
        label=f"Ra = {item['Ra']:g}"
    )

plt.xlabel("Horizontal temperature standard deviation")
plt.ylabel("Normalized height, z")
plt.title("Day 17: Horizontal temperature variation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        output_folder,
        "day17_horizontal_temperature_variation.png"
    ),
    dpi=200
)

plt.close()

# --------------------------------------------------
# 5. SAVE NUMERICAL RESULTS TO CSV
# --------------------------------------------------

csv_path = os.path.join(
    output_folder,
    "day17_temperature_profile_data.csv"
)

with open(csv_path, "w", newline="") as file:

    writer = csv.writer(file)

    writer.writerow([
        "Rayleigh_number",
        "Height_z",
        "Mean_temperature",
        "Temperature_standard_deviation",
        "Conductive_temperature",
        "Temperature_anomaly",
        "RMS_anomaly_for_run"
    ])

    for item in profiles:

        for i in range(len(item["z"])):

            writer.writerow([
                item["Ra"],
                item["z"][i],
                item["T_mean"][i],
                item["T_std"][i],
                item["T_conduction"][i],
                item["T_anomaly"][i],
                item["rms_anomaly"]
            ])

# --------------------------------------------------
# 6. PRINT SUMMARY
# --------------------------------------------------

print("\n" + "=" * 45)
print("DAY 17 ANALYSIS COMPLETED")
print("=" * 45)

print("Input folder:", result_folder)
print("Output folder:", output_folder)

print("\nRMS departure from conductive profile:")

for item in profiles:
    print(
        f"Ra = {item['Ra']:g}: "
        f"{item['rms_anomaly']:.6f}"
    )

print("\nGenerated files:")
print("1. day17_vertical_temperature_profiles.png")
print("2. day17_temperature_anomalies.png")
print("3. day17_horizontal_temperature_variation.png")
print("4. day17_temperature_profile_data.csv")



Rayleigh number: 1000.0
RMS temperature departure: 0.2518384667502802
Mean temperature at bottom: 1.0
Mean temperature at top: 0.0

Rayleigh number: 2500.0
RMS temperature departure: 0.23461324072999876
Mean temperature at bottom: 1.0
Mean temperature at top: 0.0

Rayleigh number: 500.0
RMS temperature departure: 0.2892066986086916
Mean temperature at bottom: 1.0
Mean temperature at top: 0.0

Rayleigh number: 5000.0
RMS temperature departure: 0.24482273290310824
Mean temperature at bottom: 1.0
Mean temperature at top: 0.0

DAY 17 ANALYSIS COMPLETED
Input folder: combined_mantle_results_2
Output folder: day17_temperature_profiles

RMS departure from conductive profile:
Ra = 500: 0.289207
Ra = 1000: 0.251838
Ra = 2500: 0.234613
Ra = 5000: 0.244823

Generated files:
1. day17_vertical_temperature_profiles.png
2. day17_temperature_anomalies.png
3. day17_horizontal_temperature_variation.png
4. day17_temperature_profile_data.csv
